# Extra 2 - Word2Vec CBOW do zero

No modulo 1 do curso intermediario voce treinou um **skip-gram** (a partir da
palavra alvo, prever o contexto). Aqui e o irmao dele: **CBOW** (Continuous
Bag of Words) — a partir da **media dos vetores do contexto**, prever a
palavra do meio.

Serve para praticar a mesma sintaxe (gerar pares, `nn.Embedding`, loop de
treino em PyTorch) num formato um pouco diferente.

Tente resolver antes de olhar o `_solucoes`.

In [ ]:
import re
import json
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)

corpus = [
    "the meeting is scheduled for tomorrow morning",
    "can you send me the meeting notes",
    "let us grab lunch after the meeting",
    "i will call you after the meeting tomorrow",
    "the project meeting was moved to friday",
    "please bring your notes to the meeting",
    "win a free cash prize now",
    "claim your free prize today",
    "you won a free cash award",
    "free entry to win a prize",
    "urgent claim your cash prize now",
    "call now to claim your free reward",
    "congratulations you won a free voucher",
    "reply now to claim the cash prize",
]

## 2.1 Tokenizar e montar o vocabulario

1. `tokens` = cada frase de `corpus` em minusculas, dividida por espacos
   (uma lista de listas).
2. Junte todas as palavras, pegue as unicas ordenadas.
3. `word2idx` = dicionario palavra -> indice, comecando em 0
   (nao precisa de `<PAD>` aqui).
4. `idx2word` = o inverso. Guarde `vocab_size`.

In [3]:
# 2.1
tokens = [frase.lower().split() for frase in corpus]


vocab = sorted({w for frase in tokens for w in frase })
word2idx = { w : i for i, w in enumerate(vocab)}
idx2word = {i : w for w, i in word2idx.items()}
vocab_size = len(word2idx)
print("vocab_size", vocab_size)


vocab_size 44


## 2.2 Gerar amostras CBOW

Com `window = 2`: para cada posicao `i` da frase que tenha **as 2 palavras
de cada lado** (ou seja, `window <= i < len(frase) - window`):

- `contexto` = os indices das `2*window` palavras ao redor (sem a do meio)
- `alvo` = indice da palavra na posicao `i`

Guarde em `amostras` como tuplas `(lista_de_4_indices, indice_alvo)`.
Imprima quantas amostras sairam e a primeira delas (com as palavras).

In [6]:
# 2.2
window = 2
amostras = []
for frase in tokens:
    idxs = [word2idx[w] for w in frase]
    for i in range(window, len(idxs) - window ):
        contexto = idxs[i - window:i] + idxs[i + 1: i + window + 1]
        amostras.append((contexto, idxs[i]))

print("totoal de amostras: ", len(amostras))
ctx, alvo = amostras[0]
print("Exemplo: ", [idx2word[j] for j in ctx], "alvo -> ", idx2word[alvo] )


totoal de amostras:  36
Exemplo:  ['the', 'meeting', 'scheduled', 'for'] alvo ->  is


## 2.3 Tensores de treino

- `X` = tensor `long` de formato `(n_amostras, 2*window)` com os contextos.
- `y` = tensor `long` de formato `(n_amostras,)` com os alvos.

Imprima os dois `.shape`.

In [7]:
# 2.3
X = torch.tensor([c for c, _ in amostras], dtype= torch.long)
y = torch.tensor([a for _, a in amostras], dtype = torch.long)
print("X: ", X.shape, "| Y: ", y.shape)

X:  torch.Size([36, 4]) | Y:  torch.Size([36])


## 2.4 Modelo CBOW

Uma classe `CBOW(nn.Module)`:

- `self.embedding = nn.Embedding(vocab_size, embed_dim)` com `embed_dim = 16`
- `self.linear = nn.Linear(embed_dim, vocab_size)`
- `forward(x)`: `x` chega como `(batch, 2*window)`.
  1. `self.embedding(x)` -> `(batch, 2*window, embed_dim)`
  2. media no eixo do contexto (`dim=1`) -> `(batch, embed_dim)`
  3. `self.linear(...)` -> logits `(batch, vocab_size)`

In [ ]:
# 2.4
embed_dim = 16

class CBOW(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.linear = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        vecs = self.embedding(x)
        media = vecs.mean(dim = 1)
        return self.linear(media)
    
model = CBOW(vocab_size,embed_dim)
print(model)




CBOW(
  (embedding): Embedding(44, 16)
  (linear): Linear(in_features=16, out_features=44, bias=True)
)


## 2.5 Loop de treino

- `loss_fn = nn.CrossEntropyLoss()`
- `optimizer = torch.optim.Adam(model.parameters(), lr=0.01)`
- 300 epocas, batch unico (o dataset e minusculo): `zero_grad` -> forward ->
  `loss` -> `backward` -> `step`. Imprima a loss a cada 50 epocas.

In [ ]:
# 2.5

## 2.6 Palavras parecidas

1. `E = model.embedding.weight.detach().numpy()` (matriz `(vocab_size, embed_dim)`).
2. Funcao `mais_parecidas(palavra, topn=4)`: similaridade de cosseno entre o
   vetor de `palavra` e todos os outros, retornando as `topn` maiores
   (sem contar a propria palavra).
3. Teste com `"meeting"` e `"free"`. Base minuscula = vizinhos ruidosos;
   veja so se separa mais o grupo "ham" do grupo "spam".

In [ ]:
# 2.6

## 2.7 Salvar

Salve `E` em `cbow_embeddings.npy` (`np.save`) e `word2idx` em
`cbow_vocab.json` (`json.dump`). O Extra 4 pode reaproveitar como matriz
"pre-treinada".

In [ ]:
# 2.7

## Resumo

- CBOW: contexto -> media dos vetores -> `Linear` -> prever a palavra do meio.
- Skip-gram (modulo 1) faz o caminho inverso.
- Os dois produzem a mesma coisa util no fim: a matriz `embedding.weight`.